In [ ]:
GPU="0"
num_GPUs = 1
gen_image_batch_size=256

In [ ]:
import os
os.environ['PATH'] += ':/home/temp0/anaconda3/envs/edm2/bin'

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

def show_images_from_dir(folder_path, num_images=4, start_seed=42):
    """
    顯示指定資料夾中的圖片（根據檔名排序，從指定種子碼開始），以 2x2 格顯示。

    Args:
        folder_path (str): 圖片所在資料夾路徑
        num_images (int): 要顯示的圖片數量（預設 4）
        start_seed (int): 從第幾個種子（圖片）開始（根據檔名排序）
    """
    if not os.path.isdir(folder_path):
        print(f"[錯誤] 資料夾不存在：{folder_path}")
        return

    # 過濾圖片檔案，並根據檔名中的數字排序
    image_files = sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ], key=lambda x: int(''.join(filter(str.isdigit, os.path.basename(x))) or 0))

    # 找到起始 index
    start_index = next((i for i, f in enumerate(image_files)
                        if int(''.join(filter(str.isdigit, os.path.basename(f))) or 0) >= start_seed), None)

    if start_index is None:
        print(f"[警告] 找不到 seed >= {start_seed} 的圖片。")
        return

    subset = image_files[start_index:start_index + num_images]
    if not subset:
        print(f"[警告] 從 seed {start_seed} 起沒有足夠圖片可顯示。")
        return

    # 顯示 2x2 圖片
    rows = cols = 2
    plt.figure(figsize=(8, 8))
    for i, img_path in enumerate(subset):
        img = Image.open(img_path)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# V7 Using scheduler open prob, close prob

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.0
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=3.0
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 2.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 1.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.4
close_prob = 0.3
hold_steps = 2
w_skewed = 0.38
balance_coeff = 1.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 16
w_interval_high_steps = 22
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.4
close_prob = 0.3
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.75
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.75
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.125
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 2.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.5
hold_steps = 2
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.7
close_prob = 0.6
hold_steps = 2
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.125
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.7
close_prob = 0.6
hold_steps = 2
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 目前最好V7 5萬張

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# V9:在特定Interval 降低open_prob,close_prob一個定量balance_coeff

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 100
adaptive_prob = 0.5
open_prob = 0.7
close_prob = 0.5
hold_steps = 2
w_skewed = 0.38
balance_coeff = 0.5
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v9 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# V10 把V7 mode 2 改成線性遞減

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.7
close_prob = 0.5
hold_steps = 5
w_skewed = 0.38
balance_coeff = 0.5
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 3
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 昨晚剛開始跑的

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 3
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 3
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 16
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v10_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.5
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.6
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'

show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 23
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 1.672 使用mid activate

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.0
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 14
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 16
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 22
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 20
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 22
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.35
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.45
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.25
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 50000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 以下做一些 MAP Adaptive V10

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
balance_coeff = 5.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 15
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 4
w_skewed = 0.38
balance_coeff = 6.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive_v10 \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.375
w_interval_low_steps = 16
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_const_MAP_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_x0pred \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.25
w_interval_low_steps = 16
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_const_MAP_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_x0pred \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.125
w_interval_low_steps = 16
w_interval_high_steps = 21
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_const_MAP_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_x0pred \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


w=2.125
w_interval_low_steps = 16
w_interval_high_steps = 21
num_images = 100
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 4
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_const_MAP_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_ddg_x0pred_checking \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.0
balance_coeff = 10.0
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.0
balance_coeff = 9.0
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.0
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.0
balance_coeff = 9.5
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.0
balance_coeff = 10.5
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.0
balance_coeff = 8.5
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.0
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.01
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 50000
open_prob = 0.0
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 50000
open_prob = 0.01
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0827_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.01
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.02
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.03
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.04
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.04
balance_coeff = 12.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.04
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.04
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.04
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.05
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.06
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 10000
open_prob = 0.07
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v11_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v11 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:


num_images = 100
open_prob = 0.07
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v12_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v12 \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
w_interval_low_sigma = 0.28
w_interval_high_sigma = 2.9
min_guidance = 1.75
max_guidance = 2.8
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v12_CFG_v9,w_interval_low_sigma={w_interval_low_sigma},w_interval_high_sigma={w_interval_high_sigma},min_guidance={min_guidance},max_guidance={max_guidance}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v12 \
--guidance_scheduler=const_scheduler \
--w_interval_low_sigma={w_interval_low_sigma} \
--w_interval_high_sigma={w_interval_high_sigma} \
--min_guidance={min_guidance} \
--max_guidance={max_guidance} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
w_interval_low_sigma = 0.28
w_interval_high_sigma = 2.9
min_guidance = 1.75
max_guidance = 2.5
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v12_CFG_v9,w_interval_low_sigma={w_interval_low_sigma},w_interval_high_sigma={w_interval_high_sigma},min_guidance={min_guidance},max_guidance={max_guidance}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v12 \
--guidance_scheduler=const_scheduler \
--w_interval_low_sigma={w_interval_low_sigma} \
--w_interval_high_sigma={w_interval_high_sigma} \
--min_guidance={min_guidance} \
--max_guidance={max_guidance} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 100
open_prob=0.00
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,w_interval_low_sigma={w_interval_low_sigma},w_interval_high_sigma={w_interval_high_sigma},min_guidance={min_guidance},max_guidance={max_guidance}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)

In [ ]:
num_images = 10000
open_prob=0.00
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.00
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.01
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.01
balance_coeff = 12.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.0
balance_coeff = 12.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.01
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0911_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 11.0
dirname=f'/data/guidance-team/out/2025_0911_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 12.0
dirname=f'/data/guidance-team/out/2025_0911_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0911_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 14.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 15.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 16.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 17.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.01
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.01
balance_coeff = 12.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.0
balance_coeff = 12.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.02
balance_coeff = 12.5
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
open_prob=0.02
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0909_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.02
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0912_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.005
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0912_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
# !CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
# --standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
# --preset=edm2-img512-s-guid-fid \
# --outdir=/{dirname} \
# --subdirs \
# --seeds=0-{num_images+10} \
# --sampler=cfg_adaptive_v13 \
# --guidance_scheduler=const_scheduler \
# --balance_coeff={balance_coeff} \
# --open_prob={open_prob} \
# --classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
# --debug=False \
# --heun_guid=True \
# --batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 50000
open_prob=0.015
balance_coeff = 13.0
dirname=f'/data/guidance-team/out/2025_0912_adaptive_v13_CFG_v9,open_prob={open_prob},balance_coeff={balance_coeff}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_v13 \
--guidance_scheduler=const_scheduler \
--balance_coeff={balance_coeff} \
--open_prob={open_prob} \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 跑XXL曲線用的

In [ ]:
num_images = 10000
dirname=f'/data/guidance-team/out/2025_0911_NoGuid_CFG_v9'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-xxl-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=noguid_stats \
--guidance_scheduler=const_scheduler \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--save_classifier_stats=True \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
w = 1.2
dirname=f'/data/guidance-team/out/2025_0911_const_CFG_v9_w={w}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-xxl-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_stats \
--guidance={w} \
--guidance_scheduler=const_scheduler \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--save_classifier_stats=True \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
num_images = 10000
w = 2.0
w_interval_low_sigma = 0.19
w_interval_high_sigma = 1.61
dirname=f'/data/guidance-team/out/2025_0911_interval_CFG_v9_w={w},w_interval_low_sigma={w_interval_low_sigma},w_interval_high_sigma={w_interval_high_sigma}'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-xxl-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_stats \
--guidance={w} \
--w_interval_low_sigma={w_interval_low_sigma} \
--w_interval_high_sigma={w_interval_high_sigma} \
--guidance_scheduler=interval_scheduler \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--save_classifier_stats=True \
--debug=False \
--heun_guid=True \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000